# ALS Recommendation System with Snowflake HPO API

This notebook demonstrates hyperparameter optimization for an Alternating Least Squares (ALS) recommendation system using Snowflake's Container Runtime HPO API. The implementation uses the `implicit` library for collaborative filtering on the MovieLens dataset.

## Features:
- **Multi-node Container Runtime**: Leverage Snowflake's distributed computing capabilities
- **HPO API Integration**: Use Snowflake's native hyperparameter optimization framework
- **ALS with Implicit Library**: Collaborative filtering for recommendation systems
- **Model Registry**: Automatic model registration and versioning
- **DataConnectors**: Efficient data loading and management

## Prerequisites:
- Snowflake account with Container Runtime access
- Compute pool with multi-node capability (MIN_NODES >= 2)
- External access integration to install libraries


In [ ]:
# Install required packages for ALS collaborative filtering
!pip install implicit scipy

## 1. Setup and Imports

This section initializes the environment by importing all necessary libraries for:
- Snowflake session management and ML APIs
- Data manipulation with pandas and numpy
- Sparse matrix operations for collaborative filtering
- The implicit library for ALS model training

In [ ]:
# Snowflake ML and Snowpark imports
from snowflake.snowpark.context import get_active_session
from snowflake.ml.runtime_cluster import scale_cluster, get_nodes
from snowflake.ml.modeling.tune import get_tuner_context
from snowflake.ml.modeling.tune.search import GridSearch
from snowflake.ml.modeling import tune
from snowflake.ml.data import DataConnector
from sklearn.metrics import mean_squared_error
from scipy.sparse import csr_matrix
import implicit.evaluation
import pandas as pd
import numpy as np
import time
import implicit
import requests
import zipfile
import os
import warnings
warnings.filterwarnings('ignore')

session = get_active_session()

print("✅ All imports successful!")
print(f"📦 Implicit version: {implicit.__version__}")


In [ ]:
#Set up Snowflake database, schema
database_name='ALS_RECOMMENDATION_DEMO'
schema_name='MOVIE_LENS'

def setup_environment(database_name,schema_name):
    # Create database and schema
    session.sql(f"CREATE DATABASE IF NOT EXISTS {database_name}").collect()
    session.sql(f"CREATE SCHEMA IF NOT EXISTS {database_name}.{schema_name}").collect()
    session.use_database(database_name)
    session.use_schema(schema_name)
    print(f"✅ Environment ready: {database_name}.{schema_name}")
    
setup_environment(database_name, schema_name)

In [ ]:
url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
zip_file = "/tmp/ml-1m.zip"
extract_folder = "/tmp/ml-1m"

if not os.path.exists(zip_file):
    print("Downloading MovieLens 1M dataset...")
    response = requests.get(url)
    with open(zip_file, "wb") as file:
        file.write(response.content)
    print("Download complete.")

if not os.path.exists(extract_folder):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_file, "r") as zip_ref:
        zip_ref.extractall("/tmp")
    print("Extraction complete.")

ratings_path = os.path.join(extract_folder, "ratings.dat")
column_names = ["user_id", "item_id", "rating", "timestamp"]
ratings_df = pd.read_csv(ratings_path, sep="::", names=column_names, engine='python')
rating_sdf = session.create_dataframe(ratings_df)
rating_sdf.write.save_as_table('ratings', mode='overwrite')

print(f"Dataset: {len(ratings_df):,} ratings")
print(f"Users: {ratings_df['user_id'].nunique():,}, Items: {ratings_df['item_id'].nunique():,}")
ratings_df = session.table(f'{database_name}.{schema_name}.RATINGS').to_pandas()
print(ratings_df.head())

In [ ]:
def create_data_connectors(ratings_df):
    # --- 80/20 temporal split: older ratings for training, recent for testing ---
    total_count = len(ratings_df)
    ratings_df_sorted = ratings_df.sort_values('timestamp')
    split_point = ratings_df_sorted.iloc[int(total_count * 0.8) - 1]['timestamp']

    train_df = ratings_df[ratings_df['timestamp'] <= split_point].copy()
    test_df = ratings_df[ratings_df['timestamp'] > split_point].copy()

    # Rename to the column names expected by train_func
    train_df = train_df.rename(columns={'user_id': 'USER', 'item_id': 'PRODUCT', 'rating': 'RATING'})
    test_df = test_df.rename(columns={'user_id': 'USER', 'item_id': 'PRODUCT', 'rating': 'RATING'})

    # --- Precompute index mappings (optimization) ---
    # These mappings convert user/item IDs to 0-based integer indices needed by the sparse matrix.
    all_users = pd.concat([train_df['USER'], test_df['USER']]).unique()
    all_items = pd.concat([train_df['PRODUCT'], test_df['PRODUCT']]).unique()
    user_to_idx = {user: idx for idx, user in enumerate(sorted(all_users))}
    item_to_idx = {item: idx for idx, item in enumerate(sorted(all_items))}

    # Embed the precomputed indices directly into the DataFrames
    train_df['USER_IDX'] = train_df['USER'].map(user_to_idx)
    train_df['PRODUCT_IDX'] = train_df['PRODUCT'].map(item_to_idx)
    test_df['USER_IDX'] = test_df['USER'].map(user_to_idx)
    test_df['PRODUCT_IDX'] = test_df['PRODUCT'].map(item_to_idx)

    # Store matrix dimensions so train_func knows the sparse matrix shape
    n_users = len(all_users)
    n_items = len(all_items)
    train_df['N_USERS'] = n_users
    train_df['N_ITEMS'] = n_items
    test_df['N_USERS'] = n_users
    test_df['N_ITEMS'] = n_items

    print(f"Train: {len(train_df):,} ratings")
    print(f"Test: {len(test_df):,} ratings")
    print(f"Users: {n_users}, Items: {n_items}")
    print("Columns:", train_df.columns.tolist())

    return train_df, test_df, user_to_idx, item_to_idx

train_data, test_data, user_to_idx, item_to_idx = create_data_connectors(ratings_df)

# Build reverse mappings: index -> real ID (for translating model output back)
idx_to_user = {v: k for k, v in user_to_idx.items()}
idx_to_item = {v: k for k, v in item_to_idx.items()}

print(f"Mappings: {len(user_to_idx)} users, {len(item_to_idx)} items")

# Wrap as DataConnectors for the Snowflake Tuner
dataset_map = {
    "train": DataConnector.from_dataframe(session.create_dataframe(train_data)),
    "test": DataConnector.from_dataframe(session.create_dataframe(test_data)),
}


## 2. Model Training with Hyperparameter Optimization

This section defines and executes the ALS model training with HPO:
- **Training Function**: Defines the ALS model training logic with configurable hyperparameters
- **Grid Search**: Explores combinations of RANK, REGPARAM, MAXITER, and ALPHA
- **Evaluation**: Uses accuracy metric to identify the best configuration

In [ ]:
def train_func():
    tuner_context = get_tuner_context()
    config = tuner_context.get_hyper_params()
    dm = tuner_context.get_dataset_map()

    train_pdf = dm["train"].to_pandas()
    test_pdf = dm["test"].to_pandas()

    n_users = int(train_pdf['N_USERS'].iloc[0])
    n_items = int(train_pdf['N_ITEMS'].iloc[0])

    train_user_idx = train_pdf['USER_IDX'].values.astype(int)
    train_item_idx = train_pdf['PRODUCT_IDX'].values.astype(int)
    train_ratings = train_pdf['RATING'].values.astype(np.float32)

    test_user_idx = test_pdf['USER_IDX'].values.astype(int)
    test_item_idx = test_pdf['PRODUCT_IDX'].values.astype(int)
    test_ratings = test_pdf['RATING'].values.astype(np.float32)

    user_item_train = csr_matrix(
        (train_ratings, (train_user_idx, train_item_idx)),
        shape=(n_users, n_items),
        dtype=np.float32
    )

    user_item_test = csr_matrix(
        (test_ratings, (test_user_idx, test_item_idx)),
        shape=(n_users, n_items),
        dtype=np.float32
    )

    factors = config['RANK']
    regularization = config['REGPARAM']
    iterations = config['MAXITER']
    alpha = config['ALPHA']

    als_model = implicit.als.AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        iterations=iterations,
        random_state=42
    )

    confidence_matrix = (user_item_train * alpha).astype(np.float32)
    als_model.fit(confidence_matrix)

    K = 10
    metrics = implicit.evaluation.ranking_metrics_at_k(
        als_model, user_item_train, user_item_test, K=K
    )

    p_at_k = metrics["precision"]
    map_at_k = metrics["map"]
    ndcg_at_k = metrics["ndcg"]

    print(f'Precision@{K}: {p_at_k:.4f}, MAP@{K}: {map_at_k:.4f}, NDCG@{K}: {ndcg_at_k:.4f}')

    tuner_context.report(
        metrics={
            "precision_at_10": p_at_k,
            "map_at_10": map_at_k,
            "ndcg_at_10": ndcg_at_k
        },
        model=als_model
    )

In [ ]:
# ========================== SINGLE-NODE / PARALLEL HPO ==========================
# Single node cluster and run 81 trials with 2 concurrent (we only have 3 nodes) 
print("=" * 60)
print("PHASE 1: Single-Node Parallel HPO")
print("=" * 60)

actual_nodes = len(get_nodes())
target_nodes = 1
if actual_nodes != 1:
    scale_cluster(expected_cluster_size=target_nodes)
    actual_nodes = len(get_nodes())        
print(f"Cluster scaled to {actual_nodes} node")

tuner_single = tune.Tuner(
    train_func=train_func,
    search_space={
        "RANK": [20, 50, 100],
        "REGPARAM": [0.01, 0.05, 0.1],
        "MAXITER": [10, 20, 30],
        "ALPHA": [1.0, 10.0, 40.0]
    },
    tuner_config=tune.TunerConfig(
        metric="map_at_10",
        mode="max",
        search_alg=GridSearch(),
        num_trials=81,
        max_concurrent_trials=2,
    ),
)

start_single = time.time()
tuner_results_single = tuner_single.run(dataset_map=dataset_map)
end_single = time.time()
single_node_time = end_single - start_single

print(f"\nSingle-node HPO completed in {single_node_time:.1f} seconds ({single_node_time/60:.1f} min)")
print(f"Nodes used: {actual_nodes}, Concurrent trials: 2")
print(f"Best MAP@10: {tuner_results_single.best_result['map_at_10'].values[0]:.4f}")


# Multi-node HPO

In [ ]:
# ========================== MULTI-NODE / PARALLEL HPO ==========================
# Scale cluster to 3 nodes and run 81 trials with 10 concurrent
print("=" * 60)
print("PHASE 2: Multi-Node Parallel HPO")
print("=" * 60)

target_nodes = 4
try:
    print(f"Scaling cluster to {target_nodes} nodes...")
    scale_cluster(expected_cluster_size=target_nodes)
    actual_nodes = len(get_nodes())
    print(f"Cluster scaled to {actual_nodes} node(s)")
except Exception as e:
    actual_nodes = len(get_nodes())
    print(f"Could not scale to {target_nodes} nodes (compute pool capacity).")
    print(f"Continuing with {actual_nodes} node(s) and parallel trials.")

print(f"81 trials, max_concurrent_trials=10 (parallel across {actual_nodes} nodes)")

tuner_multi = tune.Tuner(
    train_func=train_func,
    search_space={
        "RANK": [20, 50, 100],
        "REGPARAM": [0.01, 0.05, 0.1],
        "MAXITER": [10, 20, 30],
        "ALPHA": [1.0, 10.0, 40.0]
    },
    tuner_config=tune.TunerConfig(
        metric="map_at_10",
        mode="max",
        search_alg=GridSearch(),
        num_trials=81,
        max_concurrent_trials=11,
    ),
)

start_multi = time.time()
tuner_results = tuner_multi.run(dataset_map=dataset_map)
end_multi = time.time()
multi_node_time = end_multi - start_multi

print(f"\nMulti-node HPO completed in {multi_node_time:.1f} seconds ({multi_node_time/60:.1f} min)")
print(f"Nodes used: {actual_nodes}, Concurrent trials: 10")
print(f"Best MAP@10: {tuner_results.best_result['map_at_10'].values[0]:.4f}")

In [ ]:
import matplotlib.pyplot as plt

if 'single_node_time' not in dir():
    single_node_time = None

if single_node_time is not None:
    speedup = single_node_time / multi_node_time if multi_node_time > 0 else 0
    print("=" * 60)
    print("MULTI-NODE SPEEDUP COMPARISON")
    print("=" * 60)
    print(f"Single-node (1 node, sequential):    {single_node_time:>7.1f}s  ({single_node_time/60:.1f} min)")
    print(f"Multi-node  (3 nodes, 8 concurrent):  {multi_node_time:>7.1f}s  ({multi_node_time/60:.1f} min)")
    print(f"Speedup:                               {speedup:>6.1f}x")
    print("=" * 60)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    bars = ax1.bar(
        ['Single-Node\n(1 node, sequential)', 'Multi-Node\n(3 nodes, 8 concurrent)'],
        [single_node_time, multi_node_time],
        color=['#FF6B6B', '#4ECDC4'],
        width=0.5
    )
    ax1.set_ylabel('Time (seconds)', fontsize=12)
    ax1.set_title('HPO Training Time: Single vs Multi-Node (81 trials, 1M ratings)', fontsize=14, fontweight='bold')
    for bar, val in zip(bars, [single_node_time, multi_node_time]):
        ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 f'{val:.0f}s ({val/60:.1f} min)', ha='center', va='bottom', fontsize=13, fontweight='bold')
    ax2.bar(['Speedup'], [speedup], color='#45B7D1', width=0.3)
    ax2.axhline(y=1, color='gray', linestyle='--', label='Baseline (1x)')
    ax2.set_ylabel('Speedup Factor', fontsize=12)
    ax2.set_title(f'Multi-Node Speedup: {speedup:.1f}x', fontsize=14, fontweight='bold')
    ax2.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Single-node HPO was not run (skipped due to cluster scaling).")
    print(f"Multi-node HPO completed in {multi_node_time:.1f}s ({multi_node_time/60:.1f} min)")
    speedup = 0

print("\nScaling cluster back to 1 node...")
scale_cluster(expected_cluster_size=1)
print(f"Cluster size: {len(get_nodes())} node(s)")

##  Results Analysis & Experiment Tracking

Analyze HPO results and log all trials with their metrics to Snowflake Experiment Tracking.

In [ ]:
all_results = tuner_results.results
best_result = tuner_results.best_result
best_model = tuner_results.best_model

print(f"Total trials: {len(all_results)}")
print(f"Best model type: {type(best_model)}")
print(f"Available metric columns: {[c for c in all_results.columns if not c.startswith('config/')]}")
best_result

In [ ]:
# Derive full eval metrics from best trial results (already computed during HPO)
best_config = best_result.iloc[0]
full_eval_metrics = {
    'num_eval_users': len(user_to_idx),
    'full_precision_at_10': float(best_config['precision_at_10']),
    'full_map_at_10': float(best_config['map_at_10']),
    'full_ndcg_at_10': float(best_config['ndcg_at_10'])
}
print(f"Full eval: P@10={full_eval_metrics['full_precision_at_10']:.4f}, MAP@10={full_eval_metrics['full_map_at_10']:.4f}, NDCG@10={full_eval_metrics['full_ndcg_at_10']:.4f}")

# Log experiments
session.sql("CREATE OR REPLACE EXPERIMENT ALS_RECOMMENDATION_DEMO.MOVIE_LENS.ALS_RECOMMENDATION_EXPERIMENT").collect()
print("Created fresh experiment.")

from snowflake.ml.experiment import ExperimentTracking
exp = ExperimentTracking(session=session)
exp.set_experiment("ALS_RECOMMENDATION_EXPERIMENT", 
                   database_name="ALS_RECOMMENDATION_DEMO", 
                   schema_name="MOVIE_LENS")

for i, row in all_results.iterrows():
    run_name = f"hpo_trial_{i}"
    with exp.start_run(run_name):
        exp.log_params({
            "rank": int(row['config/RANK']),
            "reg_param": float(row['config/REGPARAM']),
            "max_iter": int(row['config/MAXITER']),
            "alpha": float(row['config/ALPHA']),
            "dataset": "MovieLens-1M",
            "model_type": "ALS",
            "eval_type": "sampled_500_users"
        })
      
        if 'precision_at_10' in row.index:
            exp.log_metric("precision_at_10", float(row['precision_at_10']))
        if 'map_at_10' in row.index:
            exp.log_metric("map_at_10", float(row['map_at_10']))
        if 'ndcg_at_10' in row.index:
            exp.log_metric("ndcg_at_10", float(row['ndcg_at_10']))

print(f"All {len(all_results)} HPO trials logged.")

## 3. Model Registry & Inference

Register the best ALS model in the Snowflake Model Registry as a **CustomModel**, enabling:
- SQL-callable inference (`model!recommend()`)
- Built-in versioning and lifecycle management
- RBAC governance on the model object


In [ ]:
from snowflake.ml.model import custom_model
from snowflake.ml.registry import Registry
import pickle

# Serialize the best ALS model
als_model_path = '/tmp/best_als_model.pkl'
with open(als_model_path, 'wb') as f:
    pickle.dump(best_model, f)

# Serialize the ID mappings so the model can translate real IDs <-> indices
mappings_path = '/tmp/als_mappings.pkl'
with open(mappings_path, 'wb') as f:
    pickle.dump({
        'user_to_idx': user_to_idx,
        'idx_to_item': idx_to_item,
        'idx_to_user': idx_to_user
    }, f)

print(f"Model serialized to {als_model_path}")
print(f"Mappings serialized to {mappings_path} ({len(user_to_idx)} users, {len(idx_to_item)} items)")

class ALSRecommendationModel(custom_model.CustomModel):
    """CustomModel that accepts real user IDs and returns real item IDs.
    Embeds the ALS model + ID mappings so inference is fully self-contained.
    """
    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)
        with open(self.context['als_model_path'], 'rb') as f:
            self.als_model = pickle.load(f)
        with open(self.context['mappings_path'], 'rb') as f:
            self.mappings = pickle.load(f)
        self.user_to_idx = self.mappings['user_to_idx']
        self.idx_to_item = self.mappings['idx_to_item']

    @custom_model.inference_api
    def recommend(self, input_df: pd.DataFrame) -> pd.DataFrame:
        results = []
        for _, row in input_df.iterrows():
            user_id = int(row['USER_ID'])
            user_idx = self.user_to_idx.get(user_id)

            if user_idx is None:
                results.append({
                    'USER_ID': user_id,
                    'RECOMMENDED_ITEMS': 'UNKNOWN_USER',
                    'SCORES': ''
                })
                continue

            item_indices, scores = self.als_model.recommend(
                user_idx, None, N=10, filter_already_liked_items=False
            )
            real_item_ids = [str(self.idx_to_item.get(int(idx), f'UNK_{idx}')) for idx in item_indices]
            results.append({
                'USER_ID': user_id,
                'RECOMMENDED_ITEMS': ','.join(real_item_ids),
                'SCORES': ','.join(f'{s:.4f}' for s in scores)
            })
        return pd.DataFrame(results)

model_context = custom_model.ModelContext(
    als_model_path=als_model_path,
    mappings_path=mappings_path
)
als_custom_model = ALSRecommendationModel(model_context)

# Test with real user IDs from the ML-1M dataset
test_input = pd.DataFrame({'USER_ID': [1, 100, 500, 2000, 5000]})
local_output = als_custom_model.recommend(test_input)
print("\nLocal test with real user IDs:")
local_output

In [ ]:
from snowflake.ml.model import model_signature

reg = Registry(session=session, database_name=database_name, schema_name=schema_name)

sig = model_signature.infer_signature(
    input_data=test_input,
    output_data=local_output
)

model_metrics = {
    "precision_at_10": full_eval_metrics['full_precision_at_10'],
    "map_at_10": full_eval_metrics['full_map_at_10'],
    "ndcg_at_10": full_eval_metrics['full_ndcg_at_10'],
    "num_eval_users": full_eval_metrics['num_eval_users'],
    "eval_type": "full_test_set",
    "multi_node_training_time_s": round(multi_node_time, 1),
    "dataset": "MovieLens-1M",
    "num_hpo_trials": 81,
    "num_users": len(user_to_idx),
    "num_items": len(idx_to_item)
}

if single_node_time is not None:
    model_metrics["single_node_training_time_s"] = round(single_node_time, 1)
    model_metrics["speedup"] = round(single_node_time / multi_node_time, 1) if multi_node_time > 0 else 0

mv = reg.log_model(
    als_custom_model,
    model_name="ALS_RECOMMENDATION_MODEL",
    #version_name="V2",
    signatures={"recommend": sig},
    pip_requirements=["implicit", "scipy"],
    target_platforms=["SNOWPARK_CONTAINER_SERVICES"],
    metrics=model_metrics,
    comment="ALS collaborative filtering model trained with Snowflake HPO on MovieLens 1M. Metrics from full test set evaluation."
)

print(f"Model registered: {database_name}.{schema_name}.ALS_RECOMMENDATION_MODEL (V2)")
print(f"Metrics: {mv.show_metrics()}")
print(f"Functions: {mv.show_functions()}")

In [ ]:
# ========================== SAMPLE INFERENCE ==========================
print("=" * 60)
print("SAMPLE INFERENCE: Top-10 Recommendations (Real User IDs)")
print("=" * 60)

loaded_model = mv.load(force=True)

# Use real user IDs from the MovieLens 1M dataset
sample_users = pd.DataFrame({'USER_ID': [2000, 3500, 5000, 1500, 4200]})
recommendations = loaded_model.recommend(sample_users)

print("\nRecommendations from registered model:")
for _, row in recommendations.iterrows():
    items = row['RECOMMENDED_ITEMS'].split(',')
    scores = row['SCORES'].split(',')
    print(f"\n  User {row['USER_ID']}:")
    for i, (item, score) in enumerate(zip(items[:5], scores[:5])):
        print(f"    #{i+1} Item {item:>5} (score: {score})")

print("\n" + "=" * 60)
print("WHAT'S EMBEDDED IN THE MODEL:")
print("=" * 60)
print("  1. Trained ALS user/item factor matrices (from implicit)")
print("  2. user_to_idx mapping: real user ID -> matrix index")
print("  3. idx_to_item mapping: matrix index -> real item ID")
print("  No sparse matrix needed — factors are the learned output of training.")
